# 05 - Visualisations Avancees

Ce notebook montre comment creer des visualisations du graphe.

## Outils utilises
- NetworkX pour la manipulation
- Matplotlib pour les graphiques
- PyVis pour les graphes interactifs

In [ ]:
import os
from neo4j import GraphDatabase
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

In [ ]:
NEO4J_URI = os.environ.get('NEO4J_URI', 'bolt://neo4j:7687')
driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(os.environ.get('NEO4J_USER', 'neo4j'),
          os.environ['NEO4J_PASSWORD'])
)

def query(cypher):
    with driver.session() as session:
        return [dict(r) for r in session.run(cypher)]

## 1. Construire un sous-graphe avec NetworkX

In [ ]:
# Extraire un sous-graphe d'infections
data = query("""
MATCH (p1:Patient)-[r:INFECTED_BY]->(p2:Patient)
RETURN p1.patient_id AS source, p2.patient_id AS target
LIMIT 100
""")

G = nx.DiGraph()
for row in data:
    G.add_edge(row['source'], row['target'])

print(f"Graphe: {G.number_of_nodes()} noeuds, {G.number_of_edges()} aretes")

In [ ]:
# Visualisation simple
plt.figure(figsize=(12, 8))
pos = nx.spring_layout(G, k=0.5)
nx.draw(G, pos, node_size=50, node_color='steelblue', 
        edge_color='gray', alpha=0.7, arrows=True, arrowsize=10)
plt.title('Reseau de contamination (echantillon)')
plt.tight_layout()
plt.show()

## 2. Distribution des cas par pays

In [ ]:
geo_data = query("""
MATCH (p:Patient)-[:LIVES_IN]->(l:Location)
MATCH (p)-[:HAS_DISEASE]->(:Disease)
RETURN l.country AS pays, count(p) AS cas
ORDER BY cas DESC
""")

df_geo = pd.DataFrame(geo_data)
plt.figure(figsize=(10, 6))
plt.barh(df_geo['pays'], df_geo['cas'], color='coral')
plt.xlabel('Nombre de cas')
plt.title('Distribution geographique des infections')
plt.tight_layout()
plt.show()

## 3. Distribution par variant

In [ ]:
variant_data = query("""
MATCH (p:Patient)-[:HAS_DISEASE]->(d:Disease)
RETURN d.variant AS variant, count(p) AS cas
ORDER BY cas DESC
""")

df_var = pd.DataFrame(variant_data)
plt.figure(figsize=(10, 6))
colors = plt.cm.viridis([i/len(df_var) for i in range(len(df_var))])
plt.pie(df_var['cas'], labels=df_var['variant'], autopct='%1.1f%%', colors=colors)
plt.title('Repartition par variant')
plt.show()

In [ ]:
driver.close()